# Voice dataset prep: choose a speaker, dial in pitch/bass, then export for RVC training

Three stages:
1. Preview one short clip from **every** speaker in the dataset, to pick which voice to start from
2. Once you've picked one, preview pitch/bass adjustments on that single clip -- tweak and
   re-run this cell as many times as you like, hearing before/after each time
3. Once you're happy with the values, process that speaker's **full** set of files with them,
   ready to become the training notebook's input dataset

Run this before the training notebook. Save its final output as a Kaggle Dataset
("New Dataset" from notebook output), then attach that to the training notebook.

In [ ]:
!pip install -q pedalboard soundfile huggingface_hub

## Configure

`REPO_ID`: the Hugging Face dataset repo to pull from. Check the repo's "Files" tab on
huggingface.co first -- folder layout varies between VCTK mirrors.

In [ ]:
import os

REPO_ID = "CSTR-Edinburgh/vctk"  # swap for whichever VCTK mirror you're using

PREVIEW_DIR = "/kaggle/working/previews"
RAW_DIR = "/kaggle/working/raw"
OUTPUT_DIR = "/kaggle/working/adjusted"

os.makedirs(PREVIEW_DIR, exist_ok=True)

## Stage 1: one clip per speaker

Lists every file in the dataset (cheap -- just a file listing, not a download), groups
them by speaker ID, and downloads only ONE utterance per speaker for you to listen to.

In [ ]:
import re

from huggingface_hub import HfApi, hf_hub_download

api = HfApi()
all_files = api.list_repo_files(repo_id=REPO_ID, repo_type="dataset")

# group by speaker ID (folder named like "p225"), keeping one representative
# utterance per speaker
speaker_example_path = {}
for path in all_files:
    if not path.lower().endswith((".wav", ".flac")):
        continue
    match = re.search(r"(p\d+)/", path)
    if match:
        speaker_example_path.setdefault(match.group(1), path)

print(f"found {len(speaker_example_path)} speakers")

In [ ]:
from IPython.display import Audio, display

speaker_local_path = {}
for speaker_id, repo_path in sorted(speaker_example_path.items()):
    local_path = hf_hub_download(
        repo_id=REPO_ID, repo_type="dataset", filename=repo_path, local_dir=PREVIEW_DIR
    )
    speaker_local_path[speaker_id] = local_path
    print(speaker_id)
    display(Audio(local_path))

## Stage 2: pick a speaker, tweak pitch/bass, preview before/after

Set `CHOSEN_SPEAKER` to whichever speaker ID sounded closest to what you want above.
Then tweak `PITCH_SEMITONES` / `BASS_GAIN_DB` and **re-run this cell** as many times as
you like -- each run plays the original and the adjusted version so you can compare.

In [ ]:
import soundfile as sf
from pedalboard import Pedalboard, PitchShift, LowShelfFilter

CHOSEN_SPEAKER = "p225"  # pick from the speaker IDs printed in Stage 1

PITCH_SEMITONES = -3.0  # positive = higher, negative = lower -- tweak and re-run
BASS_GAIN_DB = 4.0      # positive = boost bass, negative = cut it -- tweak and re-run


def build_board(pitch_semitones, bass_gain_db):
    effects = []
    if pitch_semitones:
        effects.append(PitchShift(semitones=pitch_semitones))
    if bass_gain_db:
        effects.append(LowShelfFilter(cutoff_frequency_hz=200.0, gain_db=bass_gain_db))
    return Pedalboard(effects)


preview_path = speaker_local_path[CHOSEN_SPEAKER]
audio, sample_rate = sf.read(preview_path, dtype="float32")  # mono: 1D array
board = build_board(PITCH_SEMITONES, BASS_GAIN_DB)
processed = board(audio.reshape(1, -1), sample_rate)[0]

print(f"{CHOSEN_SPEAKER} -- original:")
display(Audio(audio, rate=sample_rate))
print(f"{CHOSEN_SPEAKER} -- pitch={PITCH_SEMITONES:+.1f} semitones, bass={BASS_GAIN_DB:+.1f}dB:")
display(Audio(processed, rate=sample_rate))

## Stage 3: process the full chosen speaker's dataset

Only run this once you're happy with `PITCH_SEMITONES`/`BASS_GAIN_DB` from Stage 2 --
this downloads and processes every file for `CHOSEN_SPEAKER`, not just the one preview clip.

In [ ]:
from pathlib import Path

from huggingface_hub import snapshot_download

speaker_folder = str(Path(speaker_example_path[CHOSEN_SPEAKER]).parent) + "/"
print(f"Downloading all files under: {speaker_folder}")

snapshot_download(
    repo_id=REPO_ID,
    repo_type="dataset",
    allow_patterns=[f"{speaker_folder}*"],
    local_dir=RAW_DIR,
)

In [ ]:
output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

raw_speaker_dir = Path(RAW_DIR) / speaker_folder
audio_files = sorted(list(raw_speaker_dir.glob("*.wav")) + list(raw_speaker_dir.glob("*.flac")))
if not audio_files:
    raise RuntimeError(f"No .wav/.flac files found under {raw_speaker_dir}")

board = build_board(PITCH_SEMITONES, BASS_GAIN_DB)

for i, path in enumerate(audio_files, 1):
    audio, sample_rate = sf.read(str(path), dtype="float32")
    processed = board(audio.reshape(1, -1), sample_rate)[0]
    sf.write(str(output_dir / path.name), processed, sample_rate)
    if i % 25 == 0 or i == len(audio_files):
        print(f"[{i}/{len(audio_files)}] processed")

print(f"Done: {len(audio_files)} file(s) written to {output_dir}")

## Next step

Use Kaggle's "Save Version" then create a **New Dataset** from this notebook's output
(`/kaggle/working/adjusted`) -- then attach that dataset to your Applio training notebook
instead of the raw source dataset.